# Objective 

- Prepare MIMIC-III data for ML by cleaning tables and establishing a dataset suitable for predicting per admission mortality. 

## Prediction Window 

- Predictions are made using information available during the first 24 hours of the admission. 
- Target is 'hospital_expire_flag'
- Information that occurs after the first 24 hours will not be used as predictors to prevent data leakage. 
- Final dataset must contain one row per admission. 

## Database Exploration

### Importing and inspecting datasets

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

pd.set_option("display.max_columns", None)


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "mimic-iii-clinical-database-demo-1.4"
    / "mimic-iii-clinical-database-demo-1.4"
)

print(DATA_DIR)
print(DATA_DIR.exists())

/Users/tmali/Documents/Documents/MSc Healthcare Data Prep/Project - Predictive Health Analytics & Machine Learning with Python/data/raw/mimic-iii-clinical-database-demo-1.4/mimic-iii-clinical-database-demo-1.4
True


In [3]:
labevents = pd.read_csv(DATA_DIR / "LABEVENTS.csv")
admissions = pd.read_csv(DATA_DIR / "ADMISSIONS.csv")
patients = pd.read_csv(DATA_DIR / "PATIENTS.csv")
dlabitems = pd.read_csv(DATA_DIR / "D_LABITEMS.csv")
diagnoses = pd.read_csv(DATA_DIR / "DIAGNOSES_ICD.csv")
icd_dictionary = pd.read_csv(DATA_DIR / "D_ICD_DIAGNOSES.csv")

In [4]:
admissions.head()

,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,12258,10006,142345,2164-10-23 21:09:00,2164-11-01 17:15:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,SEPARATED,BLACK/AFRICAN AMERICAN,2164-10-23 16:43:00,2164-10-23 23:00:00,SEPSIS,0,1
1,12263,10011,105331,2126-08-14 22:32:00,2126-08-28 18:59:00,2126-08-28 18:59:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Private,NaN,CATHOLIC,SINGLE,UNKNOWN/NOT SPECIFIED,NaN,NaN,HEPATITIS B,1,1
2,12265,10013,165520,2125-10-04 23:36:00,2125-10-07 15:13:00,2125-10-07 15:13:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,NaN,UNKNOWN/NOT SPECIFIED,NaN,NaN,SEPSIS,1,1
3,12269,10017,199207,2149-05-26 17:19:00,2149-06-03 18:42:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,2149-05-26 12:08:00,2149-05-26 19:45:00,HUMERAL FRACTURE,0,1
4,12270,10019,177759,2163-05-14 20:43:00,2163-05-15 12:00:00,2163-05-15 12:00:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,NaN,NaN,ALCOHOLIC HEPATITIS,1,1


In [5]:
labevents.head()

,row_id,subject_id,hadm_id,itemid,charttime,value,valuenum,valueuom,flag
0,6244563,10006,NaN,50868,2164-09-24 20:21:00,19,19.0,mEq/L,NaN
1,6244564,10006,NaN,50882,2164-09-24 20:21:00,27,27.0,mEq/L,NaN
2,6244565,10006,NaN,50893,2164-09-24 20:21:00,10.0,10.0,mg/dL,NaN
3,6244566,10006,NaN,50902,2164-09-24 20:21:00,97,97.0,mEq/L,NaN
4,6244567,10006,NaN,50912,2164-09-24 20:21:00,7.0,7.0,mg/dL,abnormal


In [6]:
patients.head()

,row_id,subject_id,gender,dob,dod,dod_hosp,dod_ssn,expire_flag
0,9467,10006,F,2094-03-05 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,1
1,9472,10011,F,2090-06-05 00:00:00,2126-08-28 00:00:00,2126-08-28 00:00:00,NaN,1
2,9474,10013,F,2038-09-03 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,1
3,9478,10017,F,2075-09-21 00:00:00,2152-09-12 00:00:00,NaN,2152-09-12 00:00:00,1
4,9479,10019,M,2114-06-20 00:00:00,2163-05-15 00:00:00,2163-05-15 00:00:00,2163-05-15 00:00:00,1


In [7]:
dlabitems.head()

,row_id,itemid,label,fluid,category,loinc_code
0,1,50800,SPECIMEN TYPE,BLOOD,BLOOD GAS,NaN
1,2,50801,Alveolar-arterial Gradient,Blood,Blood Gas,19991-9
2,3,50802,Base Excess,Blood,Blood Gas,11555-0
3,4,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas,1959-6
4,5,50804,Calculated Total CO2,Blood,Blood Gas,34728-6


In [8]:
diagnoses.head()

,row_id,subject_id,hadm_id,seq_num,icd9_code
0,112344,10006,142345,1,99591
1,112345,10006,142345,2,99662
2,112346,10006,142345,3,5672
3,112347,10006,142345,4,40391
4,112348,10006,142345,5,42731


In [9]:
icd_dictionary.head()

,row_id,icd9_code,short_title,long_title
0,1,01716,Erythem nod tb-oth test,Erythema nodosum with hypersensitivity reactio...
1,2,01720,TB periph lymph-unspec,"Tuberculosis of peripheral lymph nodes, unspec..."
2,3,01721,TB periph lymph-no exam,"Tuberculosis of peripheral lymph nodes, bacter..."
3,4,01722,TB periph lymph-exam unk,"Tuberculosis of peripheral lymph nodes, bacter..."
4,5,01723,TB periph lymph-micro dx,"Tuberculosis of peripheral lymph nodes, tuberc..."


In [10]:
labevents[["hadm_id", "charttime"]].head()

,hadm_id,charttime
0,NaN,2164-09-24 20:21:00
1,NaN,2164-09-24 20:21:00
2,NaN,2164-09-24 20:21:00
3,NaN,2164-09-24 20:21:00
4,NaN,2164-09-24 20:21:00


### Checking data type and missing entries

In [11]:
labevents["charttime"].dtype

<StringDtype(storage='python', na_value=nan)>

In [12]:
labevents["charttime"].isna().sum()

np.int64(0)

In [13]:
admissions["admittime"].dtype

<StringDtype(storage='python', na_value=nan)>

In [14]:
admissions["admittime"].isna().sum()

np.int64(0)

### Converting to datetime data type and finding missing entries

In [15]:
admissions["admittime"] = pd.to_datetime(admissions["admittime"])

In [16]:
labevents["charttime"] = pd.to_datetime(labevents["charttime"])

In [17]:
admissions["admittime"].dtype

dtype('<M8[us]')

In [18]:
labevents["charttime"].dtype

dtype('<M8[us]')

In [19]:
labevents["hadm_id"].isna().sum()

np.int64(14262)

In [20]:
labevents["hadm_id"].notna().sum()

np.int64(61812)

In [21]:
labevents_clean = labevents[labevents["hadm_id"].notna()].copy()

In [22]:
labevents_clean.shape

(61812, 9)

In [23]:
labevents_clean["hadm_id"].isna().sum()

np.int64(0)

### Laboratory events without an explicit hospital admission identifier were excluded because they could not be reliably assigned to an individual hospital admission and therefore could not be confidently aligned with the prediction window.

In [24]:
labevents_clean["hadm_id"].isin(admissions["hadm_id"]).value_counts()

hadm_id
True    61812
Name: count, dtype: int64

In [25]:
labevents_clean = labevents_clean.merge(
    admissions[["hadm_id", "admittime"]],
    on= "hadm_id", 
    how = "left"
)

In [26]:
labevents_clean[["hadm_id", "admittime", "charttime"]].head()

,hadm_id,admittime,charttime
0,142345.0,2164-10-23 21:09:00,2164-10-23 17:33:00
1,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00
2,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00
3,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00
4,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00


In [27]:
labevents_clean["admittime"].isna().sum()

np.int64(0)

### Calculating 24-hour window status

In [28]:
labevents_clean["time_since_admission"] = (
    labevents_clean["charttime"] - labevents_clean["admittime"]
)

In [29]:
labevents_clean["time_since_admission"].describe()

count                      61812
mean     11 days 04:43:39.801009
std      19 days 22:11:18.166847
min            -1 days +11:45:00
25%              1 days 03:13:00
50%              4 days 11:06:00
75%             12 days 04:09:00
max            123 days 08:54:00
Name: time_since_admission, dtype: object

In [30]:
(labevents_clean["time_since_admission"] < pd.Timedelta(0)).sum()

np.int64(5172)

In [31]:
(
    (labevents_clean["time_since_admission"] >= pd.Timedelta(0)) &
    (labevents_clean["time_since_admission"] <= pd.Timedelta(hours=24))
).sum()

np.int64(9673)

### Investigating labevent entries recorded before admission

In [32]:
negative_labs = labevents_clean[
    labevents_clean["time_since_admission"] < pd.Timedelta(0)
].copy()

In [33]:
negative_labs[
    ["subject_id", "hadm_id", "admittime", "charttime", "time_since_admission", "itemid"]
].head(20)

,subject_id,hadm_id,admittime,charttime,time_since_admission,itemid
0,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:33:00,-1 days +20:24:00,50813
1,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50861
2,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50862
3,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50863
4,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50868
5,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50878
6,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50882
7,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50885
8,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50889
9,10006,142345.0,2164-10-23 21:09:00,2164-10-23 17:38:00,-1 days +20:29:00,50893


In [34]:
negative_labs["time_since_admission"].describe()

count                        5172
mean     -1 days +20:50:37.412994
std        0 days 02:11:36.344750
min             -1 days +11:45:00
25%             -1 days +20:00:00
50%             -1 days +21:15:00
75%             -1 days +22:10:15
max             -1 days +23:58:00
Name: time_since_admission, dtype: object

In [35]:
negative_labs_info = negative_labs.merge(
    admissions[["hadm_id", "admission_type", "admission_location"]],
    on="hadm_id",
    how="left"
)

In [36]:
negative_labs_info["admission_type"].value_counts()

admission_type
EMERGENCY    5088
ELECTIVE       84
Name: count, dtype: int64

In [37]:
negative_labs_info["admission_location"].value_counts()

admission_location
EMERGENCY ROOM ADMIT         3953
TRANSFER FROM HOSP/EXTRAM     579
CLINIC REFERRAL/PREMATURE     472
PHYS REFERRAL/NORMAL DELI     138
TRANSFER FROM SKILLED NUR      30
Name: count, dtype: int64

In [38]:
negative_labs_info.groupby("admission_type")["hadm_id"].nunique()

admission_type
ELECTIVE       2
EMERGENCY    107
Name: hadm_id, dtype: int64

## Dataset formation

### Creating a within 24-hour window labevents dataset

In [39]:
labevents_24h = labevents_clean[
    (labevents_clean["time_since_admission"] >= pd.Timedelta(0)) & 
    (labevents_clean["time_since_admission"] <= pd.Timedelta(hours=24))
].copy()

In [40]:
labevents_24h.shape

(9673, 11)

In [41]:
labevents_24h["time_since_admission"].describe()

count                      9673
mean     0 days 09:44:40.988317
std      0 days 06:41:12.747050
min             0 days 00:00:00
25%             0 days 03:54:00
50%             0 days 08:57:00
75%             0 days 15:10:00
max             0 days 23:58:00
Name: time_since_admission, dtype: object

In [42]:
labevents_24h["hadm_id"].nunique()

128

In [43]:
labevents_24h.groupby("hadm_id").size().describe()

count    128.000000
mean      75.570312
std       58.273680
min        3.000000
25%       41.000000
50%       55.500000
75%       92.250000
max      353.000000
dtype: float64

In [44]:
admissions[
    ~admissions["hadm_id"].isin(labevents_24h["hadm_id"])
][["hadm_id", "subject_id", "hospital_expire_flag"]]

,hadm_id,subject_id,hospital_expire_flag
25,160442,10067,1


In [45]:
labevents_24h["valuenum"].isna().sum()

np.int64(1391)

In [46]:
labevents_24h["valuenum"].notna().sum()

np.int64(8282)

In [47]:
labevents_24h["value"].head(20)

65     2.7
66      12
67      31
68     8.5
69     100
70     3.5
71      84
72     246
73     1.4
74     5.0
75     4.4
76     139
77      11
78    21.6
79    36.9
80    11.7
81     3.5
82    31.0
83    31.7
84      98
Name: value, dtype: str

In [48]:
labevents_24h[
    labevents_24h["valuenum"].isna()
]["value"].value_counts().head(20)

value
NEG              343
ART              184
INTUBATED         63
NORMAL            56
NONE              52
Clear             41
1+                40
TR                37
Yellow            35
0-2               34
HOLD              33
NotDone           33
RANDOM            30
OCCASIONAL        27
CONTROLLED        23
<0.01             22
LG                20
NOT INTUBATED     19
MOD               19
MIX               15
Name: count, dtype: int64

In [49]:
labevents_24h[labevents_24h["valuenum"].notna()]["itemid"].nunique()

162

In [50]:
labevents_24h[labevents_24h["valuenum"].notna()]["itemid"].value_counts().head(20)

itemid
51221    247
50971    245
50912    237
50983    236
51006    235
50882    230
50902    230
50931    228
50868    224
50960    214
50820    213
51265    208
50802    207
50804    207
50818    207
50821    207
51301    199
51222    197
51248    197
51249    197
Name: count, dtype: int64

In [51]:
top_numeric_items = (
    labevents_24h[labevents_24h["valuenum"].notna()]
    ["itemid"]
    .value_counts()
    .head(20)
    .index
)

dlabitems[dlabitems["itemid"].isin(top_numeric_items)][
    ["itemid", "label", "fluid", "category"]
]

,itemid,label,fluid,category
2,50802,Base Excess,Blood,Blood Gas
4,50804,Calculated Total CO2,Blood,Blood Gas
18,50818,pCO2,Blood,Blood Gas
20,50820,pH,Blood,Blood Gas
21,50821,pO2,Blood,Blood Gas
68,50868,Anion Gap,Blood,Chemistry
82,50882,Bicarbonate,Blood,Chemistry
102,50902,Chloride,Blood,Chemistry
112,50912,Creatinine,Blood,Chemistry
131,50931,Glucose,Blood,Chemistry


### Calculating admission coverage for top 20 numerical lab types

In [52]:
lab_coverage = (
    labevents_24h[labevents_24h["valuenum"].notna()]
    .groupby("itemid")["hadm_id"]
    .nunique()
    .sort_values(ascending=False)
)

In [53]:
lab_coverage.head(20)

itemid
50983    127
50912    127
50971    126
50882    126
50902    126
51006    126
50931    126
51301    125
51277    125
50868    125
51248    125
51249    125
51250    125
51222    125
51221    125
51265    125
51279    125
50960    119
50893    115
50970    115
Name: hadm_id, dtype: int64

In [54]:
top_coverage_items = lab_coverage.head(20).index

dlabitems[dlabitems["itemid"].isin(top_coverage_items)][
    ["itemid", "label", "fluid", "category"]
]

,itemid,label,fluid,category
68,50868,Anion Gap,Blood,Chemistry
82,50882,Bicarbonate,Blood,Chemistry
93,50893,"Calcium, Total",Blood,Chemistry
102,50902,Chloride,Blood,Chemistry
112,50912,Creatinine,Blood,Chemistry
131,50931,Glucose,Blood,Chemistry
160,50960,Magnesium,Blood,Chemistry
170,50970,Phosphate,Blood,Chemistry
171,50971,Potassium,Blood,Chemistry
183,50983,Sodium,Blood,Chemistry


### Selecting laboratory features for redundancy and mortality significance

In [55]:
selected_items = [
    50983, 50971, 50882, 50912, 51006,
    50931, 50893, 50960, 51222, 51221, 
    51277, 51301, 51265
]

labevents_24h[
    labevents_24h["itemid"].isin(selected_items)
]["itemid"].nunique()

13

In [56]:
labevents_selected = labevents_24h[
    labevents_24h["itemid"].isin(selected_items)
].copy()

In [57]:
labevents_selected = labevents_selected.sort_values(
    ["hadm_id", "itemid", "charttime"]
)

In [58]:
labevents_selected[
    ["hadm_id", "itemid", "charttime", "valuenum"]
].head(20)

,hadm_id,itemid,charttime,valuenum
7254,100375.0,50912,2129-05-02 09:00:00,1.3
7255,100375.0,50960,2129-05-02 09:00:00,2.1
7256,100375.0,50971,2129-05-02 09:00:00,4.5
7257,100375.0,50983,2129-05-02 09:00:00,131.0
7261,100375.0,51006,2129-05-02 09:00:00,23.0
53541,100969.0,50882,2142-11-27 00:38:00,21.0
53573,100969.0,50882,2142-11-27 08:58:00,24.0
53542,100969.0,50893,2142-11-27 00:38:00,8.7
53574,100969.0,50893,2142-11-27 08:58:00,8.5
53550,100969.0,50912,2142-11-27 00:38:00,1.1


#### Checking for duplicate labs

In [59]:
labevents_selected.duplicated(
    subset=["hadm_id", "itemid", "charttime"]
).sum()

np.int64(0)

#### Selecting the most recent lab measurements of each type

In [60]:
latest_labs = (
    labevents_selected
    .groupby(["hadm_id", "itemid"])
    .tail(1)
    .copy()
)

In [61]:
latest_labs.shape

(1617, 11)

In [62]:
latest_labs[
    ["hadm_id", "itemid", "charttime", "valuenum"]
].head(20)

,hadm_id,itemid,charttime,valuenum
7254,100375.0,50912,2129-05-02 09:00:00,1.3
7255,100375.0,50960,2129-05-02 09:00:00,2.1
7256,100375.0,50971,2129-05-02 09:00:00,4.5
7257,100375.0,50983,2129-05-02 09:00:00,131.0
7261,100375.0,51006,2129-05-02 09:00:00,23.0
53573,100969.0,50882,2142-11-27 08:58:00,24.0
53574,100969.0,50893,2142-11-27 08:58:00,8.5
53576,100969.0,50912,2142-11-27 08:58:00,1.1
53577,100969.0,50931,2142-11-27 08:58:00,387.0
53578,100969.0,50960,2142-11-27 08:58:00,1.9


In [63]:
latest_labs.groupby("hadm_id").size().describe()

count    127.000000
mean      12.732283
std        1.049934
min        5.000000
25%       13.000000
50%       13.000000
75%       13.000000
max       13.000000
dtype: float64

#### Identifying and investigating admission containing none of the selected labs

In [64]:
missing_lab_admission = admissions[
    ~admissions["hadm_id"].isin(latest_labs["hadm_id"])
][["hadm_id", "subject_id", "hospital_expire_flag"]]

missing_lab_admission

,hadm_id,subject_id,hospital_expire_flag
25,160442,10067,1
120,167612,43909,1


In [65]:
latest_labs.groupby("itemid")["hadm_id"].nunique().sort_values()

itemid
50893    115
50960    119
51221    125
51222    125
51265    125
51277    125
51301    125
50882    126
50931    126
50971    126
51006    126
50912    127
50983    127
Name: hadm_id, dtype: int64

In [66]:
labevents_24h[
    labevents_24h["hadm_id"].isin(
        missing_lab_admission["hadm_id"]
    )
][
    ["hadm_id", "itemid", "value", "valuenum", "charttime"]
].sort_values(["hadm_id", "charttime"])

,hadm_id,itemid,value,valuenum,charttime
57342,167612.0,50800,CENTRAL VENOUS,NaN,2152-10-09 20:16:00
57343,167612.0,50813,0.9,0.9,2152-10-09 20:16:00
57344,167612.0,50817,55,55.0,2152-10-09 20:16:00


In [67]:
admissions[
    admissions["hadm_id"].isin(missing_lab_admission["hadm_id"])
][
    [
        "hadm_id",
        "subject_id",
        "admittime",
        "dischtime",
        "admission_type",
        "admission_location",
        "diagnosis",
        "hospital_expire_flag"
    ]
]

,hadm_id,subject_id,admittime,dischtime,admission_type,admission_location,diagnosis,hospital_expire_flag
25,160442,10067,2130-10-06 01:34:00,2130-10-06 02:29:00,EMERGENCY,EMERGENCY ROOM ADMIT,S/P MOTORCYCLE ACCIDENT,1
120,167612,43909,2152-10-09 19:05:00,2152-10-09 22:33:00,EMERGENCY,EMERGENCY ROOM ADMIT,PNEUMONIA;TELEMETRY,1


In [68]:
dlabitems[
    dlabitems["itemid"].isin([50813, 50817])
][
    ["itemid", "label", "fluid", "category"]
]

,itemid,label,fluid,category
13,50813,Lactate,Blood,Blood Gas
17,50817,Oxygen Saturation,Blood,Blood Gas


In [69]:
admissions[
    admissions["hadm_id"].isin(missing_lab_admission["hadm_id"])
][
    ["hadm_id", "admittime", "deathtime", "dischtime", "hospital_expire_flag"]
]

,hadm_id,admittime,deathtime,dischtime,hospital_expire_flag
25,160442,2130-10-06 01:34:00,2130-10-06 02:29:00,2130-10-06 02:29:00,1
120,167612,2152-10-09 19:05:00,2152-10-09 22:33:00,2152-10-09 22:33:00,1


#### Confirming that the latest_lab dataset contains only 13 lab tests

In [70]:
latest_labs["itemid"].nunique()

13

### Reshaping the labs dataset for model use

In [71]:
labs_wide = latest_labs.pivot(
    index="hadm_id",
    columns="itemid",
    values="valuenum"
)

In [72]:
labs_wide.shape

(127, 13)

In [73]:
labs_wide.head()

itemid,50882,50893,50912,50931,50960,50971,50983,51006,51221,51222,51265,51277,51301
hadm_id,,,,,,,,,,,,,
100375.0,NaN,NaN,1.3,NaN,2.1,4.5,131.0,23.0,NaN,NaN,NaN,NaN,NaN
100969.0,24.0,8.5,1.1,387.0,1.9,4.1,142.0,19.0,40.6,13.5,183.0,13.7,15.9
101361.0,18.0,7.8,1.2,84.0,2.3,4.7,148.0,50.0,24.8,8.5,260.0,16.0,5.6
102203.0,22.0,9.1,2.2,111.0,2.9,3.3,138.0,64.0,27.6,9.5,111.0,14.1,6.2
103379.0,15.0,7.2,0.8,96.0,1.9,4.0,140.0,11.0,27.6,9.4,376.0,14.0,41.9


#### Investigating missingess 

In [74]:
labs_wide.isna().sum().sort_values(ascending=False)

itemid
50893    12
50960     8
51221     2
51222     2
51265     2
51277     2
51301     2
50882     1
50931     1
50971     1
51006     1
50912     0
50983     0
dtype: int64

In [75]:
calcium_missing = labs_wide[50893].isna()

calcium_missing.value_counts()

50893
False    115
True      12
Name: count, dtype: int64

In [76]:
calcium_check = admissions[
    ["hadm_id", "hospital_expire_flag"]
].set_index("hadm_id").join(
    labs_wide[50893].isna().rename("calcium_missing")
)

calcium_check.groupby("calcium_missing")["hospital_expire_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
calcium_missing,,,
False,115,37,0.321739
True,12,1,0.083333


- Calcium measurements were missing in 12 of 127 admissions. Mortality was observed in 1 of these 12 admissions (8.3%), compared with 37 of 115 admissions (32.2%) where calcium was recorded. Given the small number of admissions with missing calcium measurements, this difference should not be interpreted as evidence of a meaningful association.

In [77]:
magnesium_check = admissions[
    ["hadm_id", "hospital_expire_flag"]
].set_index("hadm_id").join(
    labs_wide[50960].isna().rename("magnesium_missing")
)

magnesium_check.groupby("magnesium_missing")["hospital_expire_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
magnesium_missing,,,
False,119,37,0.310924
True,8,1,0.125000


### Merging lab features onto admissions table

In [78]:
final_data = admissions.merge(
    labs_wide,
    on="hadm_id",
    how="left"
)

In [79]:
final_data.shape

(129, 32)

In [80]:
final_data[
    final_data["hadm_id"].isin(missing_lab_admission["hadm_id"])
][["hadm_id", "hospital_expire_flag"] + list(labs_wide.columns)]

,hadm_id,hospital_expire_flag,50882,50893,50912,50931,50960,50971,50983,51006,51221,51222,51265,51277,51301
25,160442,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,167612,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:
final_data[labs_wide.columns].isna().sum().sort_values(ascending=False)

50893    14
50960    10
51221     4
51222     4
51265     4
51277     4
51301     4
50882     3
50931     3
50971     3
51006     3
50912     2
50983     2
dtype: int64

In [82]:
labs_wide[50912].describe()

count    127.000000
mean       1.335433
std        1.333603
min        0.400000
25%        0.700000
50%        1.000000
75%        1.300000
max       10.500000
Name: 50912, dtype: float64

#### Investigating outlier variables

In [83]:
labs_wide.describe().T

,count,mean,std,min,25%,50%,75%,max
itemid,,,,,,,,
50882,126.0,24.373016,4.826152,7.0,21.25,25.0,27.00,38.0
50893,115.0,8.276522,0.778137,4.7,7.80,8.3,8.70,10.9
50912,127.0,1.335433,1.333603,0.4,0.70,1.0,1.30,10.5
50931,126.0,139.873016,57.953358,56.0,99.25,122.5,161.75,387.0
50960,119.0,2.068067,0.422248,1.2,1.80,2.0,2.30,3.7
50971,126.0,4.136508,0.638824,3.1,3.70,4.1,4.50,6.1
50983,127.0,139.590551,5.758850,121.0,136.00,140.0,142.00,159.0
51006,126.0,27.571429,20.746635,3.0,13.25,23.0,34.00,126.0
51221,125.0,31.197600,4.375232,21.8,28.10,31.2,34.10,41.8


In [84]:
labs_wide[[50912]].sort_values(50912, ascending=False).head(10)

itemid,50912
hadm_id,
193924.0,10.5
105150.0,7.3
175237.0,5.8
163189.0,4.8
121860.0,4.3
177759.0,4.0
117105.0,3.9
142345.0,3.5
132349.0,3.4


In [85]:
admissions[
    admissions["hadm_id"] == 193924
][[
    "hadm_id",
    "subject_id",
    "admittime",
    "dischtime",
    "admission_type",
    "admission_location",
    "diagnosis",
    "hospital_expire_flag"
]]

,hadm_id,subject_id,admittime,dischtime,admission_type,admission_location,diagnosis,hospital_expire_flag
50,193924,10120,2115-05-12 14:52:00,2115-05-15 02:15:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,LIVER FAILURE,1


In [86]:
labs_wide.loc[193924]

itemid
50882     30.0
50893      9.1
50912     10.5
50931    289.0
50960      1.8
50971      3.1
50983    137.0
51006     34.0
51221     23.0
51222      8.4
51265    119.0
51277     14.3
51301      9.9
Name: 193924.0, dtype: float64

- Outlier investigation: The maximum creatinine measurement was 10.5. Although this represents an extreme value relative to the rest of the dataset, investigation of the corresponding admission showed a patient admitted with liver failure who subsequently died during hospitalisation. Other laboratory measurements also showed abnormalities consistent with severe illness. The value was therefore retained rather than treated as an erroneous observation.

### Determining and adding 'Age' to dataset

In [87]:
patients["dob"].dtype

<StringDtype(storage='python', na_value=nan)>

In [88]:
admissions["admittime"].dtype

dtype('<M8[us]')

In [89]:
patients["dob"] = pd.to_datetime(patients["dob"])

In [90]:
patients["dob"].dtype

dtype('<M8[us]')

#### Creating an admission-level table containing each admission's DOB

In [91]:
admissions_age = admissions.merge(
    patients[["subject_id", "dob"]],
    on="subject_id",
    how="left"
)

In [92]:
admissions_age[["subject_id", "hadm_id", "admittime", "dob"]].head()

,subject_id,hadm_id,admittime,dob
0,10006,142345,2164-10-23 21:09:00,2094-03-05
1,10011,105331,2126-08-14 22:32:00,2090-06-05
2,10013,165520,2125-10-04 23:36:00,2038-09-03
3,10017,199207,2149-05-26 17:19:00,2075-09-21
4,10019,177759,2163-05-14 20:43:00,2114-06-20


In [93]:
admissions_age["age_days"] = (
    admissions_age["admittime"] - admissions_age["dob"]
).dt.days

In [94]:
admissions_age["age"] = admissions_age["age_days"] / 365.25

In [95]:
admissions_age[["subject_id", "hadm_id", "age"]].head(10)

,subject_id,hadm_id,age
0,10006,142345,70.633812
1,10011,105331,36.188912
2,10013,165520,87.082820
3,10017,199207,73.675565
4,10019,177759,48.898015
5,10026,103770,299.994524
6,10027,199395,82.491444
7,10029,132349,78.447639
8,10032,140372,88.008214
9,10033,157235,81.623546


In [96]:
admissions_age["age"].describe()

count    129.000000
mean      84.958651
std       61.109048
min       17.190965
25%       63.715264
50%       72.728268
75%       82.989733
max      300.035592
Name: age, dtype: float64

#### Investigating and adding de-identified ages for patients over 90

In [97]:
(admissions_age["age"] > 100).sum()

np.int64(9)

In [98]:
admissions_age[admissions_age["age"] > 100][
    ["subject_id", "hadm_id", "age"]
]

,subject_id,hadm_id,age
5,10026,103770,299.994524
11,10036,189483,299.994524
36,10094,168074,299.994524
37,10094,122928,300.035592
71,40655,126002,299.994524
91,41983,107689,299.994524
109,42458,159647,299.994524
115,43827,149950,299.994524
125,44154,174245,299.994524


In [99]:
admissions_age["age_raw"] = admissions_age["age"]

admissions_age["age"] = np.where(
    admissions_age["age"] > 100,
    90,
    admissions_age["age"]
)

In [100]:
admissions_age["age"].describe()

count    129.000000
mean      70.307552
std       16.381486
min       17.190965
25%       63.715264
50%       72.728268
75%       82.989733
max       90.000000
Name: age, dtype: float64

In [101]:
admissions_age = admissions_age.rename(
    columns={"age": "age_at_admission"}
)

In [102]:
final_data = final_data.merge(
    admissions_age[["hadm_id", "age_at_admission"]],
    on="hadm_id",
    how="left"
)

In [103]:
final_data[["hadm_id", "age_at_admission"]].head()

,hadm_id,age_at_admission
0,142345,70.633812
1,105331,36.188912
2,165520,87.082820
3,199207,73.675565
4,177759,48.898015


### Investigating admission_type distribution in final dataset

In [104]:
final_data["admission_type"].value_counts()

admission_type
EMERGENCY    119
ELECTIVE       8
URGENT         2
Name: count, dtype: int64

In [105]:
final_data.groupby("admission_type")["hospital_expire_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
admission_type,,,
ELECTIVE,8,0,0.000000
EMERGENCY,119,39,0.327731
URGENT,2,1,0.500000


In [106]:
final_data["admission_location"].value_counts()

admission_location
EMERGENCY ROOM ADMIT         81
TRANSFER FROM HOSP/EXTRAM    25
CLINIC REFERRAL/PREMATURE    12
PHYS REFERRAL/NORMAL DELI    10
TRANSFER FROM SKILLED NUR     1
Name: count, dtype: int64

In [107]:
final_data.groupby("admission_location")["hospital_expire_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
admission_location,,,
CLINIC REFERRAL/PREMATURE,12,1,0.083333
EMERGENCY ROOM ADMIT,81,26,0.320988
PHYS REFERRAL/NORMAL DELI,10,2,0.200000
TRANSFER FROM HOSP/EXTRAM,25,11,0.440000
TRANSFER FROM SKILLED NUR,1,0,0.000000


### Investigating insurance

In [108]:
final_data["insurance"].value_counts()

insurance
Medicare      98
Private       24
Medicaid       6
Government     1
Name: count, dtype: int64

In [109]:
final_data.groupby("insurance")["hospital_expire_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
insurance,,,
Government,1,1,1.000000
Medicaid,6,4,0.666667
Medicare,98,29,0.295918
Private,24,6,0.250000


#### Insurance will be excluded from the final dataset as it is not a direct measure of the patient's clinical or physiological state.

### Investigating ethnicity

In [110]:
final_data["ethnicity"].value_counts()

ethnicity
WHITE                                                       86
HISPANIC/LATINO - PUERTO RICAN                              15
UNKNOWN/NOT SPECIFIED                                       11
BLACK/AFRICAN AMERICAN                                       7
OTHER                                                        3
ASIAN                                                        2
HISPANIC OR LATINO                                           2
AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE     2
UNABLE TO OBTAIN                                             1
Name: count, dtype: int64

#### Ethnicity will be excluded from the predictive feature set because several categories have very small sample sizes, resulting in limited representation and potentially unstable model estimates.

### Investigating gender

In [111]:
patients["gender"].value_counts()

gender
F    55
M    45
Name: count, dtype: int64

In [112]:
final_data = final_data.merge(
    patients[["subject_id", "gender"]],
    on="subject_id",
    how="left"
)

In [113]:
final_data[["subject_id", "gender"]].head()

,subject_id,gender
0,10006,F
1,10011,F
2,10013,F
3,10017,F
4,10019,M


In [114]:
final_data["gender"].isna().sum()

np.int64(0)

#### Gender will be included in the final dataset as it represents a relatively even distribution.

### Investigating diagnoses

In [115]:
diagnoses.head()

,row_id,subject_id,hadm_id,seq_num,icd9_code
0,112344,10006,142345,1,99591
1,112345,10006,142345,2,99662
2,112346,10006,142345,3,5672
3,112347,10006,142345,4,40391
4,112348,10006,142345,5,42731


In [116]:
diagnoses["seq_num"].value_counts().sort_index()

seq_num
1     129
2     129
3     129
4     128
5     125
6     119
7     116
8     111
9     104
10     84
11     80
12     69
13     64
14     60
15     52
16     46
17     41
18     32
19     26
20     22
21     21
22     17
23     14
24      8
25      4
26      4
27      4
28      3
29      3
30      3
31      3
32      2
33      2
34      2
35      2
36      2
37      1
Name: count, dtype: int64

In [117]:
diagnoses.groupby("hadm_id").size().describe()

count    129.000000
mean      13.651163
std        6.465837
min        3.000000
25%        9.000000
50%       12.000000
75%       17.000000
max       37.000000
dtype: float64

In [118]:
primary_diagnoses = diagnoses[diagnoses["seq_num"] == 1]

primary_diagnoses[["hadm_id", "icd9_code"]].head(10)

,hadm_id,icd9_code
0,142345,99591
21,105331,570
27,165520,0389
36,199207,81201
50,177759,0389
64,103770,431
71,199395,4280
83,132349,0380
106,140372,81249
115,157235,24220


In [119]:
primary_diagnoses.merge(
    icd_dictionary[["icd9_code", "short_title", "long_title"]],
    on="icd9_code",
    how="left"
)[["hadm_id", "icd9_code", "short_title", "long_title"]].head(10)

,hadm_id,icd9_code,short_title,long_title
0,142345,99591,Sepsis,Sepsis
1,105331,570,Acute necrosis of liver,Acute and subacute necrosis of liver
2,165520,0389,Septicemia NOS,Unspecified septicemia
3,199207,81201,Fx surg nck humerus-clos,Closed fracture of surgical neck of humerus
4,177759,0389,Septicemia NOS,Unspecified septicemia
5,103770,431,Intracerebral hemorrhage,Intracerebral hemorrhage
6,199395,4280,CHF NOS,"Congestive heart failure, unspecified"
7,132349,0380,Streptococcal septicemia,Streptococcal septicemia
8,140372,81249,Fx lower humerus NEC-cl,Other closed fracture of lower end of humerus
9,157235,24220,Tox multnod goit no cris,Toxic multinodular goiter without mention of t...


In [120]:
primary_diagnoses["icd9_code"].nunique()

83

In [121]:
primary_diagnoses["icd9_code"].value_counts().head(10)

icd9_code
0389     15
51881     6
486       6
41071     5
4280      4
570       3
5715      3
1510      3
5990      3
431       2
Name: count, dtype: int64

In [122]:
primary_diagnoses["icd9_category"] = (
    primary_diagnoses["icd9_code"]
    .astype(str)
    .str[:3]
)

In [123]:
primary_diagnoses["icd9_category"].nunique()

63

In [124]:
primary_diagnoses["icd9_category"].value_counts().head(10)

icd9_category
038    23
428     8
410     8
518     7
486     6
570     3
571     3
151     3
599     3
852     3
Name: count, dtype: int64

In [125]:
primary_diagnoses["icd9_code"].astype(str).head(20)

0      99591
21       570
27      0389
36     81201
50      0389
64       431
71      4280
83      0380
106    81249
115    24220
127    43310
131     0389
145    99667
165    40291
174    41071
183    51881
191     1983
200     0383
227     1983
230     0389
Name: icd9_code, dtype: str

In [126]:
primary_diagnoses["icd9_code"].astype(str).str[0].value_counts().sort_index()

icd9_code
0    25
1    11
2     7
3     1
4    36
5    31
7     1
8    12
9     5
Name: count, dtype: int64

In [127]:
primary_diagnoses["icd9_code"].dtype

<StringDtype(storage='python', na_value=nan)>

In [128]:
primary_diagnoses["icd9_code"].head()

0     99591
21      570
27     0389
36    81201
50     0389
Name: icd9_code, dtype: str

In [129]:
primary_diagnoses["icd9_code"].str[0].value_counts().sort_index()

icd9_code
0    25
1    11
2     7
3     1
4    36
5    31
7     1
8    12
9     5
Name: count, dtype: int64

#### Defining clinical categories

In [130]:
def classify_icd9(code):
    code_num = int(code[:3])
    
    if 1 <= code_num <= 139:
        return "Infectious"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    elif 240 <= code_num <= 279:
        return "Endocrine_metabolic"
    elif 280 <= code_num <= 289:
        return "Blood"
    elif 290 <= code_num <= 319:
        return "Mental"
    elif 320 <= code_num <= 389:
        return "Nervous"
    elif 390 <= code_num <= 459:
        return "Circulatory"
    elif 460 <= code_num <= 519:
        return "Respiratory"
    elif 520 <= code_num <= 579:
        return "Digestive"
    elif 580 <= code_num <= 629:
        return "Genitourinary"
    elif 630 <= code_num <= 679:
        return "Pregnancy"
    elif 680 <= code_num <= 709:
        return "Skin"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif 740 <= code_num <= 759:
        return "Congenital"
    elif 760 <= code_num <= 779:
        return "Perinatal"
    elif 780 <= code_num <= 799:
        return "Symptoms_signs"
    elif 800 <= code_num <= 999:
        return "Injury_poisoning_other"
    else:
        return "Unknown"

In [131]:
primary_diagnoses["diagnosis_category"] = (
    primary_diagnoses["icd9_code"].apply(classify_icd9)
)

In [132]:
primary_diagnoses["diagnosis_category"].value_counts()

diagnosis_category
Circulatory               28
Infectious                26
Digestive                 18
Injury_poisoning_other    17
Respiratory               17
Neoplasms                 14
Genitourinary              4
Endocrine_metabolic        3
Nervous                    1
Musculoskeletal            1
Name: count, dtype: int64

#### Grouping categories with very few diagnoses

In [133]:
id_counts = primary_diagnoses["diagnosis_category"].value_counts()

primary_diagnoses["diagnosis_category_model"] = (
    primary_diagnoses["diagnosis_category"]
    .where(
        primary_diagnoses["diagnosis_category"].map(id_counts) >= 5,
        "Other"
    )
)

In [134]:
primary_diagnoses["diagnosis_category_model"].value_counts()

diagnosis_category_model
Circulatory               28
Infectious                26
Digestive                 18
Injury_poisoning_other    17
Respiratory               17
Neoplasms                 14
Other                      9
Name: count, dtype: int64

#### Calculating number of diagnoses records coded for each admission

In [135]:
num_diagnoses = (
    diagnoses
    .groupby("hadm_id")
    .size()
    .rename("num_diagnoses")
)

In [136]:
num_diagnoses.describe()

count    129.000000
mean      13.651163
std        6.465837
min        3.000000
25%        9.000000
50%       12.000000
75%       17.000000
max       37.000000
Name: num_diagnoses, dtype: float64

#### Merging diagnoses categories into final dataset

In [137]:
diagnosis_features = (
    primary_diagnoses[["hadm_id", "diagnosis_category_model"]]
    .merge(
        num_diagnoses.reset_index(),
        on="hadm_id",
        how="left"
    )
)

In [138]:
diagnosis_features.head()

,hadm_id,diagnosis_category_model,num_diagnoses
0,142345,Injury_poisoning_other,21
1,105331,Digestive,6
2,165520,Infectious,9
3,199207,Injury_poisoning_other,14
4,177759,Infectious,14


In [139]:
final_data = final_data.merge(
    diagnosis_features,
    on="hadm_id",
    how="left"
)

In [140]:
final_data[
    ["hadm_id", "diagnosis_category_model", "num_diagnoses"]
].head()

,hadm_id,diagnosis_category_model,num_diagnoses
0,142345,Injury_poisoning_other,21
1,105331,Digestive,6
2,165520,Infectious,9
3,199207,Injury_poisoning_other,14
4,177759,Infectious,14


In [141]:
final_data["diagnosis_category_model"].isna().sum()

np.int64(0)

### Final checks

In [142]:
final_data.shape

(129, 36)

In [143]:
final_data.columns.tolist()

['row_id',
 'subject_id',
 'hadm_id',
 'admittime',
 'dischtime',
 'deathtime',
 'admission_type',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'religion',
 'marital_status',
 'ethnicity',
 'edregtime',
 'edouttime',
 'diagnosis',
 'hospital_expire_flag',
 'has_chartevents_data',
 50882,
 50893,
 50912,
 50931,
 50960,
 50971,
 50983,
 51006,
 51221,
 51222,
 51265,
 51277,
 51301,
 'age_at_admission',
 'gender',
 'diagnosis_category_model',
 'num_diagnoses']

In [144]:
lab_names = {
    50882: "bicarbonate",
    50893: "calcium",
    50912: "creatinine",
    50931: "glucose",
    50960: "magnesium",
    50971: "potassium",
    50983: "sodium",
    51006: "urea_nitrogen",
    51221: "haematocrit",
    51222: "haemoglobin",
    51265: "platelet_count",
    51277: "rdw",
    51301: "white_blood_cells"
}

final_data = final_data.rename(columns=lab_names)

In [145]:
final_data.columns.tolist()

['row_id',
 'subject_id',
 'hadm_id',
 'admittime',
 'dischtime',
 'deathtime',
 'admission_type',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'religion',
 'marital_status',
 'ethnicity',
 'edregtime',
 'edouttime',
 'diagnosis',
 'hospital_expire_flag',
 'has_chartevents_data',
 'bicarbonate',
 'calcium',
 'creatinine',
 'glucose',
 'magnesium',
 'potassium',
 'sodium',
 'urea_nitrogen',
 'haematocrit',
 'haemoglobin',
 'platelet_count',
 'rdw',
 'white_blood_cells',
 'age_at_admission',
 'gender',
 'diagnosis_category_model',
 'num_diagnoses']

In [146]:
final_data.to_csv(
    "../data/processed/final_data.csv",
    index=False
)

In [147]:
final_data.shape

(129, 36)